# Quickstart for NeuralRNN

In this notebook, we will show how to train RNNs on a cognitive tasks, and how to train RNNs to reconstruct a dynamical system, by using NeuralRNN in a convenient way. Meanwhile, it will show two key functions of NeuralRNN:

1. Autonomous model configuration (`AutoConfig`) for different RNN variants.
2. Flexible Objective (`build_objective`) and training for task-based optimization and dynamical system reconstruction. 

## Train RNNs on a cognitive task

We use a canonical cognitive task, the context-dependent decision making task. In this task, a context cue instructs the subjects to attend to either motion or color evidence and report the sign of the attended coherence.

In [ ]:
# 1. dataset
from neuralrnn.data import  CognitiveTaskDataset

task_ds = CognitiveTaskDataset.from_task('mante')

In [ ]:
# Visualize the dataset
print(f'Inputs: {task_ds.inputs.shape}, Input dim: {task_ds.input_dim}')
print(f'Targets: {task_ds.targets.shape}, Target dim: {task_ds.output_dim}')
print(f'Mask: {task_ds.mask.shape}')
print(f'Number of conditions: {len(task_ds.conditions)}')

import matplotlib.pyplot as plt
import seaborn as sns
fig, axes = plt.subplots(4, 3, figsize=(6, 5), sharex=True)
for i in range(3):
    axes[0, i].plot(task_ds.inputs[i, :, :2], lw=1); axes[0, i].set_ylabel('Context input')
    axes[1, i].plot(task_ds.inputs[i, :, 2:4], lw=1); axes[1, i].set_ylabel('Motion input')
    axes[2, i].plot(task_ds.inputs[i, :, 4:], lw=1); axes[2, i].set_ylabel('Color input')
    axes[3, i].plot(task_ds.targets[i, :, :], lw=1); axes[3, i].set_ylabel('Target')
    
    context = task_ds.conditions[i]['context']
    axes[0, i].set_title(f'Trial {i+1} ({context})')

axes[0, 0].legend(['motion', 'color'], frameon=False, fontsize=8)
axes[1, 0].legend(['right', 'left'], frameon=False, fontsize=8)
axes[2, 0].legend(['right', 'left'], frameon=False, fontsize=8)
axes[3, 0].legend(['right', 'left'], frameon=False, fontsize=8)
plt.tight_layout(); plt.show()

Then, we build a RNN using `AutoConfig`. We specify a continuous-time RNN (ctrnn) with `model_type`, and pass the input/output/latent dim of the RNN. We also need to define the updating parameter of the RNN. We can see the initialized reccurent weight as follows.

In [ ]:
# 2. build a RNN
latent_dim = 50
from neuralrnn import AutoConfig, AutoModel
rnn_config = AutoConfig.for_model(
    model_type='ctrnn', 
    input_dim=task_ds.input_dim, 
    output_dim=task_ds.output_dim, 
    latent_dim=latent_dim, 
    alpha=0.2)
rnn = AutoModel.from_config(rnn_config)

# Visualize the recurrent weight matrix
fig = plt.figure(figsize=(3, 3))
W_rec = rnn.h2h.weight.detach().cpu().numpy()
sns.heatmap(W_rec, center=0, square=True, cmap='coolwarm', linecolor='lightgray')
plt.title('RNN recurrent connectivity')
plt.xlabel('Source unit')
plt.ylabel('Target unit')
# plt.tight_layout()
plt.show()

Now we can train this RNN on the previous task dataset. We build a supervised objective using the autonomous `build_objective` function. Then we pass the rnn, dataset, and objective into our trainer with training arguments.

In [ ]:
# 3. train the RNN

from neuralrnn import Trainer, TrainingArguments, build_objective
objective = build_objective('supervised', task_type='regression')
train_args = TrainingArguments(max_steps=3000, learning_rate=1e-3)
trainer = Trainer(rnn, task_ds, objective, train_args,)

history = trainer.train()

In [ ]:
# Visualize the recurrent weight matrix of the trained framework RNN
plt.figure(figsize=(3, 3))
W_rec = rnn.h2h.weight.detach().cpu().numpy()
sns.heatmap(W_rec, center=0, square=True, cmap='coolwarm', linecolor='lightgray')
plt.title('RNN recurrent connectivity$')
plt.xlabel('Source unit')
plt.ylabel('Target unit')
# plt.tight_layout()
plt.show()

In [ ]:
# RNN behavioral performance on the task
from neuralrnn.analysis import compute_psychometric
rnn_psycho = compute_psychometric(rnn, task_ds.inputs, task_ds.conditions)

# Visualization of the psychometric functions
fig, axes = plt.subplots(1, 2, figsize=(6, 2.5))
for ctx, color, ls in [('motion', 'black', '-'), ('color', 'gray', '--')]:
    curve = rnn_psycho['curves'].get((ctx, 'motion'))
    if curve is not None:
        axes[0].plot(curve.coherences * 100, curve.prob_right, color=color, ls=ls, label=ctx, marker='o', ms=3)
    curve = rnn_psycho['curves'].get((ctx, 'color'))
    if curve is not None:
        axes[1].plot(curve.coherences * 100, curve.prob_right, color=color, ls=ls, label=ctx, marker='o', ms=3)

for ax in axes:
    ax.axhline(0.5, color='gray', ls=':', lw=0.5)
    ax.set_ylim(-0.05, 1.05)
    ax.set_ylabel('Choice to right (%)')
    ax.legend(fontsize=7)
    sns.despine(ax=ax)
axes[0].set_xlabel('Motion coherence (%)')
axes[1].set_xlabel('Color coherence (%)')
fig.suptitle('RNN Psychometric Functions', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Neural trajectories of the RNN on the task

inputs = task_ds.inputs[:100]
conditions = task_ds.conditions[:100]
result = rnn(inputs)
states = result.states.detach().numpy()
output = result.outputs.detach().numpy()
print(f'The RNN states have shape: {states.shape} as (n_trials, n_time, n_units)')

In [ ]:
# Visualize the RNN trajectories
from neuralrnn import visualization as viz

def plot_color(condition):
    if condition['context'] == 'motion':
        return 'red' if condition['correct_choice'] > 0 else 'blue'
    elif condition['context'] == 'color':
        return 'green' if condition['correct_choice'] > 0 else 'purple'

fig = plt.figure(figsize=(3, 3))
ax = fig.add_subplot(111, projection='3d')
_ = viz.plot_trajectories_3d(
    states,
    colors=[plot_color(conditions[i]) for i in range(len(conditions))],
    alpha=0.2, linewidth=0.8, ax=ax, azim=45,
)

## Reconstruct the latent circuit of a dynamical system

In [ ]:
from neuralrnn.data import ReconstructionDataset
recon_ds = ReconstructionDataset.from_rnn_and_task(rnn, task_ds)
 
print(f'''Latent circuit dataset: 
inputs={recon_ds.inputs.shape}, 
targets={recon_ds.targets.shape}, 
====== new added ======
activity={recon_ds.activity.shape}
''')


In [ ]:
lc_config = AutoConfig.for_model(
    'latent_circuit', 
    input_dim=recon_ds.input_dim, 
    latent_dim=8, 
    output_dim=recon_ds.output_dim, 
    embedding_dim=recon_ds.activity.shape[-1],
    alpha=0.2,
)
lc_model = AutoModel.from_config(lc_config)

In [ ]:
lc_objective = build_objective('reconstruction', state_map='embedding')

lc_training_args = TrainingArguments(
    max_steps=16000, 
    learning_rate=2e-2, 
    weight_decay=0.001,
    log_every=200,
)

def lc_post_step_hook(model):
    model.apply_constraints()

lc_trainer = Trainer(
    lc_model, 
    recon_ds, 
    lc_objective, 
    lc_training_args,
    post_step_hook=lc_post_step_hook,
)

lc_history = lc_trainer.train() # it will take minutes to train the latent circuit model

In [ ]:
lc_result = compute_psychometric(lc_model, task_ds.inputs, task_ds.conditions)

fig, axes = plt.subplots(1, 2, figsize=(6, 2.5))
for ctx, color, ls in [('motion', 'black', '-'), ('color', 'gray', '--')]:
    curve = lc_result['curves'].get((ctx, 'motion'))
    if curve is not None:
        axes[0].plot(curve.coherences * 100, curve.prob_right, color=color, ls=ls, label=ctx)
    curve = lc_result['curves'].get((ctx, 'color'))
    if curve is not None:
        axes[1].plot(curve.coherences * 100, curve.prob_right, color=color, ls=ls, label=ctx)
for ax in axes:
    ax.axhline(0.5, color='gray', ls=':', lw=0.5)
    ax.set_ylim(-0.05, 1.05)
    ax.set_ylabel('Choice to right (%)')
    ax.legend(fontsize=7)
    sns.despine(ax=ax)
axes[0].set_xlabel('Motion coherence (%)')
axes[1].set_xlabel('Color coherence (%)')
fig.suptitle('Latent Circuit Psychometric Functions', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(7, 4), width_ratios=[2, 2, 1])

# Latent recurrent connectivity w_rec
w_rec = lc_model.w_rec.weight.data.detach()
sns.heatmap(w_rec, ax=axes[0], center=0, cmap='coolwarm', linewidths=0.5, vmin=-1, vmax=1, cbar=False, square=True)
axes[0].set_title(r'$w_{rec}$')
axes[0].set_xlabel('Source node')
axes[0].set_ylabel('Target node')

# Input connectivity w_in (input_dim x latent_dim)
w_in = lc_model.w_in.weight.data.detach()
sns.heatmap(w_in, ax=axes[1], center=0, cmap='coolwarm', linewidths=0.5, cbar=False, square=True)
axes[1].set_title(r'$w_{in}$')
axes[1].set_xlabel('Latent node')
axes[1].set_ylabel('Input channel')

# Output connectivity w_out (output_dim x latent_dim)
w_out = lc_model.w_out.weight.data.detach()
sns.heatmap(w_out.T, ax=axes[2], center=0, cmap='coolwarm', linewidths=0.5, cbar=False, square=True)
axes[2].set_title(r'$w_{out}$')
axes[2].set_xlabel('Output')
axes[2].set_ylabel('Latent node')

plt.suptitle('Latent Circuit Connectivity', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Compare the latent circuit states with the framework RNN states
inputs = task_ds.inputs[:5]
result = rnn(inputs)
y = result.states      # (N, T, latent_dim) — framework RNN hidden states
lc_out = lc_model(inputs)
x = lc_out.states       # (N, T, latent_dim) — latent circuit states
qx = lc_model.embed(x)  # (N, T, embedding_dim) — embedded latent states

fig = plt.figure(figsize=(3, 3))
ax = fig.add_subplot(111)
ax.scatter(qx.detach().flatten().numpy(), y.detach().flatten().numpy(), s=1, alpha=0.1)
lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]), max(ax.get_xlim()[1], ax.get_ylim()[1])]
ax.plot(lims, lims, 'k-', alpha=0.75, lw=1)
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel(r'$Qx$', fontsize=10)
ax.set_ylabel(r'$y$', fontsize=10)
sns.despine(ax=ax)
plt.title('Agreement: $Qx$ vs $y$', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize the RNN trajectories
inputs = task_ds.inputs[:100]
lc_states = lc_model(inputs).states

fig = plt.figure(figsize=(3, 3))
ax = fig.add_subplot(111, projection='3d')
_ = viz.plot_trajectories_3d(
    states,
    colors=[plot_color(conditions[i]) for i in range(len(conditions))],
    alpha=0.2, linewidth=0.8, ax=ax, azim=45,
)